Design patterns are categorized into three main types

—Creational, Structural, and Behavioral—

which provide reusable solutions to common software design problems. They improve code flexibility, reusability, and maintainability by defining best practices for object creation, composition, and communication between objects

### Creational

Singleton

In [ ]:
# only one instance OR single point of access
# cons: Breaks single responsepility, might result a smelly code , might cause tetabitiry issues, State of life

# singletn in one thread

class A:
    pass

print(type(A))
# type is the thing that creates classes.
#I want to create my own version of type (the thing that creates classes)


<class 'type'>


In [ ]:
from threading import Thread

def work():
    print("working...")

t = Thread(target=work)
t.start()
t.join()   #.join() = “WAIT until this thread finishes”

print("done") #out put might vary because they working at same time if join not used

working...
done


In [1]:
from threading import Thread
import time

instances = [] 

class Singlton(type):
    _instances = {}

    def __call__(self, *args, **kwargs):
        if self not in self._instances:
            time.sleep(1)
            instance = self._instances[self] = super().__call__(*args, **kwargs)
            self._instances[self] = instance
        return self._instances[self]


class Network(metaclass=Singlton):
    def log(self):
        return self


def create_sin():
    sin = Network()
    print(sin)
    instances.append(sin)
    return sin


if __name__ == '__main__':
    # s1 = create_sin()
    # s2 = create_sin()

    #print(s1 is s2)  

    # multi thread
    threads = [Thread(target=create_sin) for _ in range(2)]

    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print(instances[0] is instances[1])

# u can use lock or make the sleep one line later


<__main__.Network object at 0x00000263EFC65D00><__main__.Network object at 0x00000263EFC66750>

False


factory

In [2]:
# requiremnt changes, dynamic switching, separation of concerns, used when for example i don't care what type of data base i am utalizing i want an instance

from dataclasses import dataclass
from abc import ABC

# 1. Use a Dataclass to make the return type professional and structured
@dataclass
class Currency:
    name: str
    code: str

# 2. Base Class where the core logic lives
class Country(ABC):
    def __init__(self, fiat: Currency, virtual: Currency):
        # As you suggested: the currency is initialized within the country itself
        self.fiat = fiat
        self.virtual = virtual

    def get_info(self) -> str:
        # Implementing your "self.class" idea: 
        # self.__class__.__name__ dynamically grabs the name of the child class (e.g., "Egypt")
        # without us having to hardcode it in every single country!
        country_name = self.__class__.__name__
        return f"{country_name} -> Fiat: {self.fiat.code} | Crypto: {self.virtual.code}"

# 3. Child classes simply define their specific data in their __init__
class USA(Country):
    def __init__(self):
        super().__init__(
            fiat=Currency("US Dollar", "USD"),
            virtual=Currency("Bitcoin", "BTC")
        )

class Spain(Country):
    def __init__(self):
        super().__init__(
            fiat=Currency("Euro", "EUR"),
            virtual=Currency("Ethereum", "ETH")
        )

class Egypt(Country):
    def __init__(self):
        super().__init__(
            fiat=Currency("Egyptian Pound", "EGP"),
            virtual=Currency("Tether", "USDT")
        )

# 4. The Factories (Now 100% compliant with "Tell, Don't Ask")
class FiatCurrencyFactory:
    def create_currency(self, country: Country) -> str:
        # No if/else statements. It just asks the object for its data.
        return country.fiat.code

class VirtualCurrencyFactory:
    def create_currency(self, country: Country) -> str:
        return country.virtual.code

# --- Testing the Implementation ---
if __name__ == '__main__':
    # Instantiate our countries
    usa = USA()
    egypt = Egypt()
    spain = Spain()

    # See the dynamic self.__class__ in action
    print("--- Dynamic Self Introspection ---")
    print(usa.get_info())
    print(egypt.get_info())
    print()

    # The Factories in action
    fiat_factory = FiatCurrencyFactory()
    crypto_factory = VirtualCurrencyFactory()

    print("--- Factory Output ---")
    print(f"USA Fiat: {fiat_factory.create_currency(usa)}")
    print(f"Egypt Crypto: {crypto_factory.create_currency(egypt)}")


--- Dynamic Self Introspection ---
USA -> Fiat: USD | Crypto: BTC
Egypt -> Fiat: EGP | Crypto: USDT

--- Factory Output ---
USA Fiat: USD
Egypt Crypto: USDT


Abstract factory

In [2]:
from abc import ABC, abstractmethod

# ==========================================
# 1. THE PRODUCT BLUEPRINTS (Abstract Products)
# Concept: These are the "rules". They don't do anything 
# except force other classes to have specific methods.
# ==========================================

class NeuralNetwork(ABC):
    @abstractmethod
    def forward_pass(self, data):
        pass

class TrainingPipeline(ABC):
    @abstractmethod
    def run_epoch(self):
        pass


# ==========================================
# 2. THE ACTUAL PRODUCTS (Concrete Products)
# Concept: These are the real, working objects that 
# follow the blueprints above.
# ==========================================

class VisionNetwork(NeuralNetwork):
    def forward_pass(self, data):
        return f"Processing pixels: {data}"

class VisionPipeline(TrainingPipeline):
    def run_epoch(self):
        return "Vision model trained for 1 epoch."


# ==========================================
# 3. THE FACTORY BLUEPRINT (Abstract Factory)
# Concept: The rulebook for the factory itself. 
# It guarantees that ANY factory we build will 
# produce both a network and a pipeline.
# ==========================================

class AIFactory(ABC):
    @abstractmethod
    def create_network(self) -> NeuralNetwork:
        pass

    @abstractmethod
    def create_pipeline(self) -> TrainingPipeline:
        pass


# ==========================================
# 4. THE ACTUAL FACTORY (Concrete Factory)
# Concept: The machine that actually spits out 
# the Vision objects we defined in step 2.
# ==========================================

class VisionFactory(AIFactory):
    def create_network(self):
        return VisionNetwork()  # Spits out a VisionNetwork

    def create_pipeline(self):
        return VisionPipeline() # Spits out a VisionPipeline


# ==========================================
# 5. THE CLIENT (The User / Environment)
# Concept: This is where the magic happens. 
# This class only talks to the Blueprints, never 
# to the specific Vision classes directly.
# ==========================================

class AIEnvironment:
    def __init__(self, factory: AIFactory):
        # We hand it a factory. It blindly trusts the factory
        # to give it a matching network and pipeline.
        self.network = factory.create_network()
        self.pipeline = factory.create_pipeline()

    def run(self):
        print(self.pipeline.run_epoch())
        print(self.network.forward_pass([255, 128, 0]))

# --- RUNNING THE CODE ---
my_vision_factory = VisionFactory()
my_env = AIEnvironment(my_vision_factory)
my_env.run()

Vision model trained for 1 epoch.
Processing pixels: [255, 128, 0]


In [1]:
#One level up. no need to know what factory we use, how it created, care about using the implemntation

from abc import ABC, abstractmethod
from enum import IntEnum
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

class ModelTier(IntEnum):
    NANO = 1
    FLASH = 2
    PRO = 3
    ULTRA = 4

@dataclass
class ModelConfig:
    tier: ModelTier
    parameters_billion: float
    is_quantized: bool
    meta_tags: Optional[List[str]] = None

class NeuralNetwork(ABC):
    @abstractmethod
    def forward_pass(self, input_data: Any) -> Any:
        pass

class TrainingPipeline(ABC):
    @abstractmethod
    def run_epoch(self) -> float:
        pass

class VisionNetwork(NeuralNetwork):
    def __init__(self, config: ModelConfig):
        self.config = config

    def forward_pass(self, input_data: List[float]) -> List[float]:
        return [pixel * 0.5 for pixel in input_data]

class LanguageNetwork(NeuralNetwork):
    def __init__(self, config: ModelConfig):
        self.config = config

    def forward_pass(self, input_data: str) -> str:
        return input_data.upper()

class VisionPipeline(TrainingPipeline):
    def run_epoch(self) -> float:
        return 0.15

class LanguagePipeline(TrainingPipeline):
    def run_epoch(self) -> float:
        return 0.08

class AIFactory(ABC):
    @abstractmethod
    def create_network(self, config: ModelConfig) -> NeuralNetwork:
        pass

    @abstractmethod
    def create_pipeline(self) -> TrainingPipeline:
        pass

class VisionFactory(AIFactory):
    def create_network(self, config: ModelConfig) -> NeuralNetwork:
        return VisionNetwork(config)

    def create_pipeline(self) -> TrainingPipeline:
        return VisionPipeline()

class LanguageFactory(AIFactory):
    def create_network(self, config: ModelConfig) -> NeuralNetwork:
        return LanguageNetwork(config)

    def create_pipeline(self) -> TrainingPipeline:
        return LanguagePipeline()

class AIProductionEnvironment:
    def __init__(self, factory: AIFactory, config: ModelConfig):
        self.network = factory.create_network(config)
        self.pipeline = factory.create_pipeline()

    def execute_test_run(self, sample_input: Any) -> Dict[str, Any]:
        loss = self.pipeline.run_epoch()
        output = self.network.forward_pass(sample_input)
        
        return {
            "final_loss": loss,
            "inference_result": output
        }

if __name__ == '__main__':
    vision_config = ModelConfig(
        tier=ModelTier.FLASH,
        parameters_billion=1.5,
        is_quantized=True,
        meta_tags=["experimental", "fast-inference"]
    )
    
    language_config = ModelConfig(
        tier=ModelTier.PRO,
        parameters_billion=70.0,
        is_quantized=False
    )

    vision_system = AIProductionEnvironment(VisionFactory(), vision_config)
    vision_results = vision_system.execute_test_run([255.0, 128.0, 64.0])
    
    language_system = AIProductionEnvironment(LanguageFactory(), language_config)
    language_results = language_system.execute_test_run("hello world")

    print(vision_results)
    print(language_results)


{'final_loss': 0.15, 'inference_result': [127.5, 64.0, 32.0]}
{'final_loss': 0.08, 'inference_result': 'HELLO WORLD'}


Builder

In [19]:
class NetworkService:
    def __init__(self, url=None, sss=None):
        self.url = url
        self.sss = sss

    def show(self):
        for key, value in self.__dict__.items():
            print(f"{key}: {value}")


class NetworkBuilder:
    def __init__(self):
        self._url = None
        self._sss = None

    def set_url(self, url):
        self._url = url
        return self

    def set_sss(self, sss):
        self._sss = sss
        return self

    def build(self):
        service = NetworkService(url=self._url, sss=self._sss)
        self._url = None
        self._sss = None
        return service


if __name__ == '__main__':
    builder = NetworkBuilder()
    service = builder.set_url("google.com").set_sss("ss").build()
    service.show()

url: google.com
sss: ss


Prototype

In [28]:
# copying an existing object instead of creatting(maybe it went through a user interaction or so)
# u can't copy scince some names might be invisible, tight coupling(u need to know the structure of the original class)
# copy functionality, the object has the access to all internal stuff
# useful in testing

import copy
from abc import ABC , abstractmethod

class DefaultShapes(ABC):
    @abstractmethod
    def draw():
        pass

class Square(DefaultShapes):
    def __init__(self,size):
        self.size = size

    def draw(self):
        return f'drawing square of size : {self.size}'

class Circle(DefaultShapes):
    def __init__(self,radius):
        self.radius = radius

    def draw(self):
        return f'drawing circle of radius : {self.radius}'


class AbstractArt:
    def __init__(self,shapes):
        self.shapes = shapes
    
    def draw(self):
        return [x.draw() for x in self.shapes]


if __name__ == '__main__':
    shapess = [Square(4),Circle(3)]
    art = AbstractArt(shapess)
    x = art.draw()
    print(x[0])
    art2 = copy.deepcopy(art)
    y = art2.draw()
    print(y)


    



drawing square of size : 4
['drawing square of size : 4', 'drawing circle of radius : 3']


In [ ]:
from abc import ABC, abstractmethod
import copy

# ----------------------------
# 1️⃣ Singleton: Only one ZooRegistry can exist
# ----------------------------
class ZooRegistry:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ZooRegistry, cls).__new__(cls)
            cls._instance.animals = []
        return cls._instance

    def add_animal(self, animal):
        self.animals.append(animal)

# ----------------------------
# 2️⃣ Abstract Factory: Families of animals
# ----------------------------
class AnimalFactory(ABC):
    @abstractmethod
    def create_animal(self, name):
        pass

# Mammal Factory
class MammalFactory(AnimalFactory):
    def create_animal(self, name):
        return Mammal(name)

# Bird Factory
class BirdFactory(AnimalFactory):
    def create_animal(self, name):
        return Bird(name)

# ----------------------------
# 3️⃣ Factory Method: Animal subclasses define creation
# ----------------------------
class Animal(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def speak(self):
        pass

class Mammal(Animal):
    def speak(self):
        return f"{self.name} the mammal says: Hello!"

class Bird(Animal):
    def speak(self):
        return f"{self.name} the bird says: Tweet!"

# ----------------------------
# 4️⃣ Builder: Construct an Enclosure step by step
# ----------------------------
class Enclosure:
    def __init__(self):
        self.parts = []

    def show(self):
        return "Enclosure with: " + ", ".join(self.parts)

class EnclosureBuilder:
    def __init__(self):
        self.enclosure = Enclosure()

    def add_fence(self):
        self.enclosure.parts.append("fence")
        return self

    def add_water(self):
        self.enclosure.parts.append("water pond")
        return self

    def add_trees(self):
        self.enclosure.parts.append("trees")
        return self

    def build(self):
        return self.enclosure

# ----------------------------
# 5️⃣ Prototype: Clone an existing animal
# ----------------------------
class AnimalPrototype(Animal):
    def speak(self):
        return f"{self.name} the prototype says: Roar!"
    def clone(self):
        return copy.deepcopy(self)


# ----------------------------
# Usage Example
# ----------------------------
if __name__ == "__main__":
    # Singleton: Only one zoo registry
    zoo = ZooRegistry()
    zoo2 = ZooRegistry()
    assert zoo is zoo2  # Both are the same instance

    # Abstract Factory + Factory Method: Create animals
    mammal_factory = MammalFactory()
    bird_factory = BirdFactory()

    lion = mammal_factory.create_animal("Leo")
    parrot = bird_factory.create_animal("Polly")

    zoo.add_animal(lion)
    zoo.add_animal(parrot)

    print([a.speak() for a in zoo.animals])  # Polymorphic animal behavior

    # Builder: Construct a new enclosure
    builder = EnclosureBuilder()
    enclosure = builder.add_fence().add_water().add_trees().build()
    print(enclosure.show())

    # Prototype: Clone an animal
    cloned_lion = AnimalPrototype("Simba").clone()
    #print(cloned_lion.speak())

['Leo the mammal says: Hello!', 'Polly the bird says: Tweet!']
Enclosure with: fence, water pond, trees


### STRUCTURAL DESIGN PATTERNS
Adapter


In [2]:
# Scikit-Learn vs. PyTorch/TensorFlow.

# Scikit-Learn relies on a very simple, standard interface: .fit(X, y) to train and .predict(X) to test.

# PyTorch models are highly customizable and do not have a standard .fit() method. They require you to convert data to Tensors, set up DataLoaders, and write custom loops for epochs and forward passes.

# If you have a pipeline that expects Scikit-Learn models, but you want to drop a powerful PyTorch Deep Learning model into that exact same pipeline, you need an Adapter.

In [3]:
# 1. The interface

from abc import ABC, abstractmethod

class ScikitLearnInterface(ABC):
    """The standard interface our system expects."""
    
    @abstractmethod
    def fit(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

# A standard model works perfectly with this interface natively.
class StandardRandomForest(ScikitLearnInterface):
    def fit(self, X, y):
        print("RandomForest: Fitting data natively...")
        
    def predict(self, X):
        print("RandomForest: Predicting...")
        return [0, 1, 0]

In [4]:
# 2. The adaptee
class PyTorchNeuralNet:
    """The incompatible model (Adaptee)."""
    
    def train_epochs(self, dataloader, epochs, lr):
        print(f"PyTorch Model: Training for {epochs} epochs using dataloader...")
        
    def forward_pass(self, tensor_data):
        print("PyTorch Model: Running forward pass on tensor...")
        # Returns raw probabilities instead of final class labels
        return [0.1, 0.9, 0.3]

In [5]:
# 3. The adapter
class PyTorchToScikitAdapter(ScikitLearnInterface):
    """The Adapter that bridges the gap."""
    
    def __init__(self, pytorch_model: PyTorchNeuralNet):
        # The adapter wraps the Adaptee
        self.model = pytorch_model 

    def fit(self, X, y):
        print("Adapter: Translating X, y into PyTorch DataLoader...")
        mock_dataloader = "DataLoader(X, y)" 
        
        print("Adapter: Delegating training to the PyTorch model...")
        # We translate the simple .fit() call into the complex PyTorch call
        self.model.train_epochs(mock_dataloader, epochs=10, lr=0.001)

    def predict(self, X):
        print("Adapter: Translating X into a PyTorch Tensor...")
        mock_tensor = "Tensor(X)"
        
        print("Adapter: Getting raw probabilities from PyTorch forward pass...")
        raw_probs = self.model.forward_pass(mock_tensor)
        
        print("Adapter: Translating probabilities back to standard class labels...")
        # E.g., apply a threshold of 0.5 to convert probabilities to 0 or 1
        return [1 if prob >= 0.5 else 0 for prob in raw_probs]

In [6]:
def evaluate_model(model: ScikitLearnInterface, X_train, y_train, X_test):
    """The client code expects standard .fit() and .predict() methods."""
    print(f"\n--- Evaluating {model.__class__.__name__} ---")
    
    # The pipeline only speaks the standard language
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    print(f"Final pipeline predictions: {predictions}")

# 1. Using a standard model
rf_model = StandardRandomForest()
evaluate_model(rf_model, X_train="data", y_train="labels", X_test="data")

# 2. Using the incompatible model via the Adapter
# We initialize the PyTorch model, then wrap it in the Adapter
pytorch_model = PyTorchNeuralNet()
adapted_pytorch_model = PyTorchToScikitAdapter(pytorch_model)

# The pipeline accepts the adapted PyTorch model flawlessly!
evaluate_model(adapted_pytorch_model, X_train="data", y_train="labels", X_test="data")


--- Evaluating StandardRandomForest ---
RandomForest: Fitting data natively...
RandomForest: Predicting...
Final pipeline predictions: [0, 1, 0]

--- Evaluating PyTorchToScikitAdapter ---
Adapter: Translating X, y into PyTorch DataLoader...
Adapter: Delegating training to the PyTorch model...
PyTorch Model: Training for 10 epochs using dataloader...
Adapter: Translating X into a PyTorch Tensor...
Adapter: Getting raw probabilities from PyTorch forward pass...
PyTorch Model: Running forward pass on tensor...
Adapter: Translating probabilities back to standard class labels...
Final pipeline predictions: [0, 1, 0]


Bridge

In [8]:
# separation of concerns doesn't happen here


from abc import ABC, abstractmethod

# ==========================================
# 1. The Implementor Hierarchy (The "Bridge" destination)
# ==========================================
class Color(ABC):
    @abstractmethod
    def fill_color(self) -> str:
        pass

class RedColor(Color):
    def fill_color(self) -> str:
        return "Red"

class BlueColor(Color):
    def fill_color(self) -> str:
        return "Blue"


# ==========================================
# 2. The Abstraction Hierarchy (The "Bridge" origin)
# ==========================================
class Shape(ABC):
    def __init__(self, color: Color):
        # THIS IS THE BRIDGE! 
        # The Shape holds a reference to a Color object.
        self.color = color 

    @abstractmethod
    def draw(self) -> str:
        pass

class Circle(Shape):
    def draw(self) -> str:
        # It delegates the color implementation to the bridged object
        return f"Drawing a Circle filled with {self.color.fill_color()}."

class Square(Shape):
    def draw(self) -> str:
        return f"Drawing a Square filled with {self.color.fill_color()}."


# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # Create our implementations (Colors)
    red = RedColor()
    blue = BlueColor()

    # Create our abstractions (Shapes) and pass the implementations across the bridge
    red_circle = Circle(red)
    blue_square = Square(blue)
    red_square = Square(red)

    print(red_circle.draw())   # Output: Drawing a Circle filled with Red.
    print(blue_square.draw())  # Output: Drawing a Square filled with Blue.
    print(red_square.draw())   # Output: Drawing a Square filled with Red.

Drawing a Circle filled with Red.
Drawing a Square filled with Blue.
Drawing a Square filled with Red.


In [9]:
# If stuff is represented as a tree

from abc import ABC, abstractmethod

# ==========================================
# 1. The Component (Common Interface)
# ==========================================
class Item(ABC):
    @property
    @abstractmethod
    def price(self) -> float:
        """Both a single product and a box of products must have a price."""
        pass

# ==========================================
# 2. The Leaf (Single Object)
# ==========================================
class Product(Item):
    def __init__(self, name: str, price: float):
        self.name = name
        self._price = price

    @property
    def price(self) -> float:
        # Just returns its own fixed price
        return self._price

# ==========================================
# 3. The Composite (Container Object)
# ==========================================
class Box(Item):
    def __init__(self):
        self.contents = [] # Can hold Products or other Boxes

    def add(self, item: Item):
        self.contents.append(item)

    @property
    def price(self) -> float:
        # Recursively sums the 'price' property of everything inside
        return sum(item.price for item in self.contents)


# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # 1. Create single products (Leaves)
    phone = Product("Smartphone", 800.0)
    charger = Product("Charger", 20.0)
    headphones = Product("Headphones", 150.0)

    # 2. Create boxes (Composites)
    small_box = Box()
    small_box.add(charger)
    small_box.add(headphones)

    big_box = Box()
    big_box.add(phone)
    big_box.add(small_box) # Putting a box inside a box!

    # 3. Get the prices using the @property
    print(f"Phone price: ${phone.price}")
    print(f"Small box price (charger + headphones): ${small_box.price}")
    
    # This automatically calculates the phone + everything inside the small box
    print(f"Big box total price: ${big_box.price}")

Phone price: $800.0
Small box price (charger + headphones): $170.0
Big box total price: $970.0


Decorator

In [20]:
# attach new behaviors to object without altering existing code
from abc import ABC, abstractmethod

class CoffeeMachine(ABC):
    @abstractmethod
    def make_small(self):
        pass
    
    def make_large(self):
        pass


class BasicCoffee(CoffeeMachine):
    def make_small(self):
        return f'making small coffee'
    
    def make_large(self):
        return f' making large coffee'


class EnahancedCoffe(CoffeeMachine):
    # passing the object for functionality
    def __init__(self, coffee: BasicCoffee):
        self.coffee = coffee

    def make_small(self):
        return self.coffee.make_small()
    
    def make_large(self):
        return f'making largeee'
    
    def milk(self):
        return f'{self.coffee.make_small()} and adding milk'

    
if __name__ == '__main__':
    basic = BasicCoffee()
    enhanced = EnahancedCoffe(basic)

    print(enhanced.make_large())
    print(enhanced.milk())





making largeee
making small coffee and adding milk


Facade

In [28]:
# Simple interface for complex functionality
from dataclasses import dataclass

class ComplexStorage:
    def __init__(self, filepath):
        self.filepath = filepath
        self.cache = {}
        print('reading data')

    def store(self, key,value):
        self.cache[key] = value

    def read(self, key):
        return self.cache[key]
    
    def commit(self):
        print(f' stroing to {self.filepath}')

@dataclass
class User:
    login: str
    nick_name: str

class Repo:
    def __init__(self):
        self.system_prefrence = ComplexStorage('/dd/dd.om')
    
    def save(self, user: User):
        self.system_prefrence.store(user.login,user.nick_name)
        return self.system_prefrence.commit()

if __name__ == '__main__':
    user_repo = Repo()
    user = User('joun.m','john')
    user_repo.save(user)





reading data
 stroing to /dd/dd.om


Flyweight

In [ ]:
# Reduce memory and processing footprint, if u have lots of similar objects


import random
from abc import ABC, abstractmethod
from enum import Enum   

# ---------------------------
# ABSTRACT BASE CLASS
# ---------------------------
class Sprite(ABC):
    @abstractmethod
    def draw(self):
        pass

    @abstractmethod
    def move(self, x: int, y: int):
        pass


# ---------------------------
# ENUM FOR RANKS 
# ---------------------------
class FighterRank(Enum):
    member = 0
    co_leader = 1
    leader = 2

# IMPORTANT:
# FighterRank.member is NOT just 0
# It is an Enum object: FighterRank.member
#
# It HAS a value:
# FighterRank.member.value == 0
#
# It also has a name:
# FighterRank.member.name == "member"


# ---------------------------
# FIGHTER CLASS (FLYWEIGHT OBJECT)
# ---------------------------
class Fighter(Sprite):
    def __init__(self, rank: FighterRank):
        # Store the Enum value (not just a number)
        self.rank = rank

    def draw(self):
        # .name gives "member", "co_leader", etc.
        print(f'drawing fighter with rank: {self.rank.name}')

    def move(self, x, y):
        print(f'moving fighter of rank: {self.rank.name} to ({x}, {y})')


# ---------------------------
# STORAGE (FLYWEIGHT FACTORY)
# ---------------------------
class Storage:
    def __init__(self):
        # Store ONE fighter per rank
        self.fighters = {}

    def get_fighter(self, rank: FighterRank):
        # If this rank was never created before → create it once
        if rank not in self.fighters:
            print(f'Creating new Fighter for rank {rank.name}')
            self.fighters[rank] = Fighter(rank)

        # Return the SAME object every time
        return self.fighters[rank]


# ---------------------------
# ARMY CLASS (CLIENT)
# ---------------------------
class Army:
    def __init__(self):
        self.army = []        # list of fighter references
        self.storage = Storage()

    def spawn_fighter(self, rank: FighterRank):
        # reuse object instead of creating new one
        fighter = self.storage.get_fighter(rank)
        self.army.append(fighter)

    def draw_army(self):
        for fighter in self.army:

            # VERY IMPORTANT EXPLANATION:
            #
            # fighter.rank is an Enum (e.g., FighterRank.member)
            #
            # match compares AGAINST Enum members, NOT numbers, NOT strings
            #
            # So we write:
            #   case FighterRank.member   correct
            #
            # NOT:
            #   case 0                   wrong
            #   case "member"           wrong

            match fighter.rank:
                case FighterRank.member:
                    print('M')   # Member
                case FighterRank.co_leader:
                    print('C')   # Co-leader
                case FighterRank.leader:
                    print('L')   # Leader


# ---------------------------
# MAIN
# ---------------------------
if __name__ == "__main__":
    army_size = 10
    army = Army()

    for _ in range(army_size):
        # random 0,1,2 → convert to Enum
        r = random.randrange(3)

        # Convert number → Enum
        # Example: 0 → FighterRank.member
        rank = FighterRank(r)

        army.spawn_fighter(rank)

    army.draw_army()



Creating new Fighter for rank co_leader
Creating new Fighter for rank leader
Creating new Fighter for rank member
C
L
M
L
M
M
M
L
L
C


Proxy

In [40]:
# Object in front of object similar to faceade but same interface, similar to decorator but manages the whole lofe cycle of original object

# A proxy acts as a "middle man" that controls access to a real object.

# School scenario:
# - Real object → GradeBook (stores student grades)
# - Proxy → controls who can access grades (e.g., only teachers can modify)


from abc import ABC, abstractmethod


# ---------------------------
# SUBJECT INTERFACE (RULES)
# ---------------------------
class GradeAccess(ABC):
    @abstractmethod
    def view_grades(self):
        pass

    @abstractmethod
    def add_grade(self, student, grade):
        pass


# ---------------------------
# REAL OBJECT (ACTUAL DATA)
# ---------------------------
class GradeBook(GradeAccess):
    def __init__(self):
        # stores student grades
        self.grades = {}

    def view_grades(self):
        # show all grades
        print("Grades:", self.grades)

    def add_grade(self, student, grade):
        # add/update grade
        self.grades[student] = grade
        print(f"Added grade {grade} for {student}")


# ---------------------------
# PROXY (CONTROLS ACCESS)
# ---------------------------
class GradeBookProxy(GradeAccess):

    _shared_gradebook = GradeBook()

    def __init__(self, role):
        # role could be "student" or "teacher"
        self.role = role


    def view_grades(self):
        # both students and teachers can view
        print(f"[{self.role}] requesting to view grades...")
        self._shared_gradebook.view_grades()

    def add_grade(self, student, grade):
        # ONLY teachers can add grades
        print(f"[{self.role}] requesting to add grade...")

        if self.role != "teacher":
            print("Access denied: only teachers can add grades")
            return

        # if allowed → forward request to real object
        self._shared_gradebook.add_grade(student, grade)


# ---------------------------
# USAGE (SCENARIO)
# ---------------------------
if __name__ == "__main__":

    # Teacher access
    teacher = GradeBookProxy("teacher")
    teacher.add_grade("Ali", 90)     
    teacher.view_grades()

    print("------")

    # Student access
    student = GradeBookProxy("student")
    student.add_grade("Ali", 100)    
    student.view_grades()

[teacher] requesting to add grade...
Added grade 90 for Ali
[teacher] requesting to view grades...
Grades: {'Ali': 90}
------
[student] requesting to add grade...
Access denied: only teachers can add grades
[student] requesting to view grades...
Grades: {'Ali': 90}


### BEHAVIOURAL DESIGN PATTERNS

Chain of responsibility

In [ ]:
# Define a chaing of handlers to process a request, decides to process or/and pass it to next handlers


from __future__ import annotations
from abc import ABC, abstractmethod


# Define a chain of handlers to process a request, decides to process or/and pass it to next handlers

from __future__ import annotations
from abc import ABC, abstractmethod

# ==========================================
# 1. The Interface
# ==========================================
class Handler(ABC):
    """The base interface that all handlers must follow."""
    
    @abstractmethod
    def set_next(self, handler: Handler) -> Handler:  #The from __future__ import annotations line tells Python: "Hey, don't read the type hints yet. Just treat them as strings for now and figure them out later."
        pass

    @abstractmethod
    def handle(self, request: str) -> str | None:
        pass


# ==========================================
# 2. The Base Handler
# ==========================================
class AbstractHandler(Handler):
    """Provides the default boilerplate for linking handlers together."""
    
    _next_handler: Handler = None

    def set_next(self, handler: Handler) -> Handler:
        # Save the next handler in the chain
        self._next_handler = handler
        # Return the next handler so we can chain them like: a.set_next(b).set_next(c)
        return handler

    @abstractmethod
    def handle(self, request: str) -> str | None:
        # If this handler can't process it, pass it to the next one (if it exists)
        if self._next_handler:
            return self._next_handler.handle(request)
        return None


# ==========================================
# 3. The Concrete Handlers
# ==========================================
class Level1Support(AbstractHandler):
    def handle(self, request: str) -> str | None:
        if request == "password_reset":
            return "Level 1: I have reset your password."
        # If it's not a password reset, pass it up the chain using super()
        return super().handle(request)

class Level2Support(AbstractHandler):
    def handle(self, request: str) -> str | None:
        if request == "bug_report":
            return "Level 2: I have logged the bug for the dev team."
        # Pass it up the chain
        return super().handle(request)

class Level3Support(AbstractHandler):
    def handle(self, request: str) -> str | None:
        if request == "server_crash":
            return "Level 3: I am rebooting the main servers now!"
        # Pass it up the chain
        return super().handle(request)


# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # 1. Create the handlers
    l1 = Level1Support()
    l2 = Level2Support()
    l3 = Level3Support()

    # 2. Link them together: Level 1 -> Level 2 -> Level 3
    l1.set_next(l2).set_next(l3)

    # 3. Send requests to the START of the chain (Level 1)
    requests = ["password_reset", "server_crash", "refund_request", "bug_report"]

    for req in requests:
        print(f"\nSending request: '{req}'")
        
        # The client ONLY interacts with the first handler in the chain
        result = l1.handle(req)
        
        if result:
            print(f"  Handled: {result}")
        else:
            print(f"  Ignored: No one in the chain could handle this.")



Sending request: 'password_reset'
  Handled: Level 1: I have reset your password.

Sending request: 'server_crash'
  Handled: Level 3: I am rebooting the main servers now!

Sending request: 'refund_request'
  Ignored: No one in the chain could handle this.

Sending request: 'bug_report'
  Handled: Level 2: I have logged the bug for the dev team.


command

In [3]:
# invoke internal functionality: button to do something, it does decoupling but it focus on encapsulation for queueing or ordering or so

from abc import ABC, abstractmethod
import queue


class Command(ABC):
    def __init__(self, command_id):
        self.command_id = command_id
    @abstractmethod
    def excute(self):
        print(f'excuting {self}')
    
class OrderFood(Command):
    def excute(self):
        print(f'ordering food: {self.command_id}')

class PayFood(Command):
    def excute(self):
        print(f'paying for food: {self.command_id}')
    

class ProcessCommand:
    queue = []

    def add_to_queue(self,command: Command):
        self.queue.append(command)

    def process(self):
        [item.excute() for item in self.queue]
        self.queue = []


if __name__ == "__main__":
    processor = ProcessCommand()
    processor.add_to_queue(OrderFood('Botato'))
    processor.add_to_queue(PayFood('Botato'))

    processor.process()



ordering food: Botato
paying for food: Botato


Interpreter

In [16]:
# usefull for processing langauge: like mathmatical equation

from ast import main


class AbstractExpression:
    @staticmethod
    def interpret(self):
        pass

class Terminal(AbstractExpression):  # this for numbers
    def __init__(self, value):
        self.value = float(value)

    def interpret(self):
        return self.value

class NonTerminal(AbstractExpression): #this for operators
    def __init__(self, left, right):
        self.left = left
        self.right = right

class Add(NonTerminal):
    def interpret(self):
        return self.left.interpret() + self.right.interpret()
    

class Sub(NonTerminal):
    def interpret(self):
        return self.left.interpret() - self.right.interpret()


if __name__ == "__main__":

    expression = '5 + 11 - 3 + 7'
    tokens = expression.split(" ")
    target = []

    for i in range(len(tokens)):
        if i == 0:
            target.append(Terminal(tokens[0]))
        elif tokens[i] == "+":
            target.append(Add(target.pop(), Terminal(tokens[i+1])))
        elif tokens[i] == "-":
            target.append(Sub(target.pop(), Terminal(tokens[i+1])))
    
    result = target.pop().interpret()
    print(result)
    print(Add(Terminal(4),Terminal(5)).interpret())
    

20.0
9.0


Iterator

In [31]:
# traerse a collection

class AlphabeticalOrder:
    def __init__(self, words, reverse: bool = False):
        self.collection = sorted(words)
        self.reverse = reverse
        self.position = len(words)-1 if reverse else 0
    
    def next(self):
        value = self.collection[self.position]
        self.position += -1 if self.reverse else 1
        return value
    
    def has_next(self):
        if self.reverse and self.position > -1:
            return True
        elif not self.reverse and self.position < len(self.collection):
            return True
        else:
            return False

class Collection:
    def __init__(self, collection):
        self.collection = collection
    
    def iter(self):
        return AlphabeticalOrder(self.collection, reverse= False)
    
    
    def reverse_iter(self):
        return AlphabeticalOrder(self.collection, reverse= True)


if __name__ == "__main__":
    collection = Collection(['ABC','Collectionmona'])
    iterator = collection.iter()
    while iterator.has_next():
        print(iterator.next())

ABC
Collectionmona


Mediator

In [39]:
# distributed system where nodes need to communicate, like microservices or chat applicaition, center server to forward

from __future__ import annotations

class ChatUser:
    mediator = None

    def __init__(self, name):
        self.name = name
    
    def set_med(self, med: Mediator):
        self.mediator = med
    
    def send(self, msg):
        print(f'{self.name} sending {msg} to every one')
        self.mediator.send_msg(msg,self)
    
    def receive(self, msg, sender: ChatUser):
        print(f'everyone receiving {msg} from {sender.name}')


class Mediator:
    users = []

    def add_user(self, user: ChatUser):
        self.users.append(user)
        user.set_med(self)
    
    def send_msg(self, msg, sender: ChatUser):
        for u in self.users:
            if u != sender:
                u.receive(msg, sender)

if __name__ == "__main__":

    mediator = Mediator()
    user1 = ChatUser('user1')
    user2 = ChatUser('user2')
    mediator.add_user(user1)
    mediator.add_user(user2)

    user1.send('hello')

user1 sending hello to every one
everyone receiving hello from user1


Momento

In [40]:
# Undo and Redo and these type of things capsulations. getting previous details
# Momento -> stores state, Originator -> creates state, Caretaker -> decides to save or restore state

from dataclasses import dataclass

@dataclass
class TextMemento:
    """
    A simple, locked data container. 
    It stores the exact state of the text editor at a specific moment in time.
    """
    text: str
    cursor_position: int

class TextEditor:
    """The actual application that does the work and contains the real data."""
    def __init__(self):
        self.text = ""
        self.cursor_position = 0

    def type_words(self, words: str):
        self.text += words
        self.cursor_position += len(words)
        print(f"User typed. Current text: '{self.text}'")

    def save_state(self) -> TextMemento:
        """Creates a snapshot of exactly how the editor looks right now."""
        return TextMemento(text=self.text, cursor_position=self.cursor_position)

    def restore_state(self, memento: TextMemento):
        """Takes a snapshot and rewrites its own internal data to match it."""
        self.text = memento.text
        self.cursor_position = memento.cursor_position
        print(f"State restored. Current text: '{self.text}'")


class UndoManager:
    """
    Decides WHEN to save and WHEN to restore. 
    It keeps a list of Mementos, but it NEVER edits the Mementos directly.
    """
    def __init__(self, editor: TextEditor):
        self.editor = editor
        self.history: list[TextMemento] = []

    def hit_save_button(self):
        print("Caretaker: Asking editor to save...")
        # 1. Ask the Originator for a Memento
        snapshot = self.editor.save_state()
        # 2. Store the Memento in the history list
        self.history.append(snapshot)

    def hit_undo_button(self):
        if not self.history:
            print("Caretaker: Nothing to undo!")
            return
            
        print("\nCaretaker: Hitting UNDO! Handing last save back to the editor...")
        # 1. Get the most recent Memento from the history
        last_snapshot = self.history.pop()
        # 2. Hand it back to the Originator to restore
        self.editor.restore_state(last_snapshot)


if __name__ == "__main__":
    editor = TextEditor()
    undo_system = UndoManager(editor)

    # 1. Type something and save it
    editor.type_words("Hello ")
    undo_system.hit_save_button() # Saves "Hello "

    # 2. Type more and save it
    editor.type_words("World! ")
    undo_system.hit_save_button() # Saves "Hello World! "

    # 3. Make a mistake (Don't save it)
    editor.type_words("jsdkfljsdf") 

    # 4. Use the Caretaker to fix the mistake
    undo_system.hit_undo_button() # Restores back to "Hello World! "
    undo_system.hit_undo_button() # Restores back to "Hello "


User typed. Current text: 'Hello '
Caretaker: Asking editor to save...
User typed. Current text: 'Hello World! '
Caretaker: Asking editor to save...
User typed. Current text: 'Hello World! jsdkfljsdf'

Caretaker: Hitting UNDO! Handing last save back to the editor...
State restored. Current text: 'Hello World! '

Caretaker: Hitting UNDO! Handing last save back to the editor...
State restored. Current text: 'Hello '


Observer

In [ ]:
# Registry to notify the result when done instead of asking for retrievel constantly, Define a subscription mechanism for delivery, One to many relationship

from abc import ABC, abstractmethod
from typing import List
from __future__ import annotations
# ==========================================
# 1. The Interfaces (The Rules)
# ==========================================
class Subscriber(ABC):
    """The Observer: Anyone who wants to be notified must follow this rule."""
    @abstractmethod
    def toggle(self, channel : YouTubeChannel):
        pass
    @abstractmethod
    def receive_notification(self, channel_name: str, video_title: str) -> None:
        pass


class Channel(ABC):
    """The Subject/Publisher: Any channel must be able to manage a list of people."""
    @abstractmethod
    def subscribe(self, subscriber: Subscriber) -> None:
        pass

    @abstractmethod
    def unsubscribe(self, subscriber: Subscriber) -> None:
        pass

    @abstractmethod
    def notify_subscribers(self, video_title: str) -> None:
        pass


# ==========================================
# 2. The Concrete Implementations
# ==========================================
class YouTubeChannel(Channel):
    """The actual channel that people subscribe to."""
    def __init__(self, channel_name: str) -> None:
        self.channel_name = channel_name
        # This is the "Registry" you mentioned in your comment!
        # It holds a list of all the active observers.
        self.subscriber_list: List[Subscriber] = []

    def subscribe(self, subscriber: Subscriber) -> None:
        self.subscriber_list.append(subscriber)
        print(f" Someone just subscribed to {self.channel_name}!")

    def unsubscribe(self, subscriber: Subscriber) -> None:
        if subscriber in self.subscriber_list:
            self.subscriber_list.remove(subscriber)
            print(f" Someone unsubscribed from {self.channel_name}.")

    def notify_subscribers(self, video_title: str) -> None:
        """This loops through the registry and alerts everyone."""
        print(f"\n {self.channel_name} is publishing a new video: '{video_title}'")
        for subscriber in self.subscriber_list:
            # We "push" the notification to them
            subscriber.receive_notification(self.channel_name, video_title)

    def upload_video(self, video_title: str) -> None:
        """A helper method to simulate uploading a video and automatically notifying."""
        # ... logic to actually upload the video to servers goes here ...
        
        # Once it's done, notify everyone automatically!
        self.notify_subscribers(video_title)


class YouTubeUser(Subscriber):
    """The actual user who receives the notifications."""
    def __init__(self, username: str) -> None:
        self.username = username
        self.muted_channels = set()

    def toggle(self, channel : YouTubeChannel):
        if channel.channel_name in self.muted_channels:
            self.muted_channels.remove(channel.channel_name)
        else:
            self.muted_channels.add(channel.channel_name)


    def receive_notification(self, channel_name: str, video_title: str) -> None:
        # This is what happens when the channel taps the user on the shoulder
        if channel_name not in self.muted_channels:
            print(f"  -> {self.username}'s Phone Buzzes: {channel_name} uploaded '{video_title}'")



if __name__ == "__main__":
    tech_channel = YouTubeChannel("TechWithTim")

    user_alice = YouTubeUser("Alice")
    user_bob = YouTubeUser("Bob")
    user_charlie = YouTubeUser("Charlie")

    print("--- Setting up subscriptions ---")
    tech_channel.subscribe(user_alice)
    tech_channel.subscribe(user_bob)

    tech_channel.upload_video("Python Design Patterns in 10 Minutes")

    print("\n--- Changing subscriptions ---")
    tech_channel.unsubscribe(user_bob)
    tech_channel.subscribe(user_charlie)
    user_charlie.toggle(tech_channel)

    tech_channel.upload_video("Understanding the Observer Pattern!")

 


--- Setting up subscriptions ---
 Someone just subscribed to TechWithTim!
 Someone just subscribed to TechWithTim!

 TechWithTim is publishing a new video: 'Python Design Patterns in 10 Minutes'
  -> Alice's Phone Buzzes: TechWithTim uploaded 'Python Design Patterns in 10 Minutes'
  -> Bob's Phone Buzzes: TechWithTim uploaded 'Python Design Patterns in 10 Minutes'

--- Changing subscriptions ---
 Someone unsubscribed from TechWithTim.
 Someone just subscribed to TechWithTim!

 TechWithTim is publishing a new video: 'Understanding the Observer Pattern!'
  -> Alice's Phone Buzzes: TechWithTim uploaded 'Understanding the Observer Pattern!'


State

In [80]:
# Object behavious change based on internal state, State can is capsulated in an objectd
from __future__ import annotations
from abc import ABC, abstractmethod

class AITrainingJob:
    def __init__(self) -> None:
        self.current_state: PipelineState = DataPrepState()
        
    def set_state(self, state: PipelineState) -> None:
        self.current_state = state

    def process_data(self) -> None:
        self.current_state.process_data(self)

    def train(self) -> None:
        self.current_state.train(self)

    def deploy(self) -> None:
        self.current_state.deploy(self)


class PipelineState(ABC):
    @abstractmethod
    def process_data(self, job: AITrainingJob) -> None:
        pass

    @abstractmethod
    def train(self, job: AITrainingJob) -> None:
        pass

    @abstractmethod
    def deploy(self, job: AITrainingJob) -> None:
        pass


class DataPrepState(PipelineState):
    def process_data(self, job: AITrainingJob) -> None:
        print("Cleaning data...")
        job.set_state(TrainingState())

    def train(self, job: AITrainingJob) -> None:
        print("Can't do that, data isn't ready.")

    def deploy(self, job: AITrainingJob) -> None:
        print("Can't do that, data isn't ready.")


class TrainingState(PipelineState):
    def process_data(self, job: AITrainingJob) -> None:
        print("Can't do that, data is already prepped.")

    def train(self, job: AITrainingJob) -> None:
        print("Training model...")
        job.set_state(DeployableState())

    def deploy(self, job: AITrainingJob) -> None:
        print("Can't do that, model isn't trained yet.")


class DeployableState(PipelineState):
    def process_data(self, job: AITrainingJob) -> None:
        print("Already finished and deployed!")

    def train(self, job: AITrainingJob) -> None:
        print("Already finished and deployed!")

    def deploy(self, job: AITrainingJob) -> None:
        print("Model deployed to production!")


if __name__ == "__main__":
    my_model_job = AITrainingJob()

    print("--- Trying to deploy early ---")
    my_model_job.deploy()

    print("\n--- Doing it right ---")
    my_model_job.process_data()
    my_model_job.train()
    my_model_job.deploy()

--- Trying to deploy early ---
Can't do that, data isn't ready.

--- Doing it right ---
Cleaning data...
Training model...
Model deployed to production!
